In [158]:
'''
!pip install exifread
!pip install requests
!pip install datetime
!pip install Pillow

from PIL import Image, ImageDraw, ImageFont
import exifread
import requests
import datetime
'''


def get_address_from_image(image_path):
  # Extract GPS data from EXIF
  with open(image_path, 'rb') as f:
    tags = exifread.process_file(f)
    latitude = tags.get('GPS GPSLatitude')
    longitude = tags.get('GPS GPSLongitude')
    latitude_ref = tags.get('GPS GPSLatitudeRef')
    longitude_ref = tags.get('GPS GPSLongitudeRef')
    orientation_ref = tags.get('Image Orientation')
    date_time_original = tags.get('EXIF DateTimeOriginal')

  #Tags = piexif.dump(tags)
  # Check if GPS data is available
  if not latitude or not longitude or not date_time_original:
    return None
  #print(tags)
  date_obj = datetime.strptime(str(date_time_original), "%Y:%m:%d %H:%M:%S")
  date_print = " (" + date_obj.strftime("%b %d,'%y") +")"
  #print(date_print)

  # Convert coordinates to decimal degrees
  latitude = convert_to_decimal(latitude)
  longitude = convert_to_decimal(longitude)

  # Set north/south and east/west based on reference tags
 # latitude_dir = 'N' if latitude_ref == 'N' else 'S'
 # longitude_dir = 'E' if longitude_ref == 'E' else 'W'

   # Fix + - for the latitude / longuitude
  if str(latitude_ref) == "S":
    latitude = -latitude
  if str(longitude_ref) == "W":
    longitude = -longitude

  # Use geocoding API to get address
  api_key = 'YOUR API KEY'  # Replace with your API key
  url = f"https://maps.googleapis.com/maps/api/geocode/json?latlng={latitude},{longitude}&key={api_key}"
  response = requests.get(url)
  data = response.json()
  #print(data)
    # Extract and return address if successful
  # Extract address components if successful
  ##print(data['results'])
  if data['status'] == 'OK':
    city = ""
    state = ""
    county = ""
    country = "*"
    for result in data['results']:
      # Use address_components for more precise extraction
      for component in result['address_components']:
        if component['types'][0] == 'administrative_area_level_1':
          state = component['short_name']
        elif component['types'][0] == 'country':
          country = component['long_name']
        elif component['types'][0] == 'administrative_area_level_2':
          county = component['long_name']
        elif component['types'][0] == 'locality':
          city = component['long_name']
    # Use formatted_address as fallback
    if city != "":
       city = city+", "
    if state != "":
       state = state+", "
    if county != "":
       county = county
    if country != "*":
       country_a = country 
    else:
       country_a = ""
    ##address = f"{city}{county}{country_a}"

   
     ######## CITY MORE IMPORTANT
    if city!= "":
        address = [f"{city}",f"{country_a}{date_print}"]
    elif county != "":
        address = [f"{county}",f"{country_a}{date_print}"]
    elif state != "":
        address = [f"{state}",f"{country_a}{date_print}"]
    elif country != "*":
        address = [f"{country}",f"{date_print}"]
        
     ##### COUNTY MORE IMPORTANT #######
    if country == "Egypt" or country == "Indonesia":
        if county != "":
            address = [f"{county}",f"{country_a}{date_print}"]
        elif state != "":
            address = [f"{state}",f"{country_a}{date_print}"]
        elif country != "*":
            address = [f"{county}",f"{date_print}"]
            
         ##### STATE  MORE IMPORTANT #######   
    if country == "Indonesia":
        if state != "":
            address = [f"{state}",f"{country_a}{date_print}"]
        elif country != "*":
            address = [f"{county}",f"{date_print}"]
    if country == "Japan" or country == "India" or country == "Chile" or country == "Australia" or country == "Philippines" or country == "Vietnam" or country == "Türkiye" or country == "Tanzania" or country == "Malaysia":
        if county != "":
            address = [f"{city}{county}",f"{country_a}{date_print}"]
        else:
            address = [f"{city}{state}",f"{country_a}{date_print}"]
        
    print(address)



    #print(f"{city}, {county}, {state}, {country} {date_print}")
    return address, country , orientation_ref

  else:
    orientation_ref = "Rotated 90 CW"
    last_folder_name = image_path.parent.name
    words = last_folder_name.split()
    country = words[1]
    date_created = os.path.getctime(image_path)
    date_print = time.strftime("%m/%d/%y", time.localtime(date_created))
    address = [f"{country}", f"{date_print}"]
    ##print(f"{city}, {county}, {country}")
    return address, country , orientation_ref



def convert_to_decimal(coord):
  degrees = float(coord.values[0].num) / float(coord.values[0].den)
  minutes = float(coord.values[1].num) / float(coord.values[1].den)
  seconds = float(coord.values[2].num) / float(coord.values[2].den)
  return degrees + minutes / 60 + seconds / 3600

'''
# Example usage
image_path = '/content/drive/MyDrive/Image processing/IMG_8556.JPG'
address = get_address_from_image(image_path)

if address:
  print(address)
else:
  print("No GPS data found in image or geocoding failed.")
'''

'\n# Example usage\nimage_path = \'/content/drive/MyDrive/Image processing/IMG_8556.JPG\'\naddress = get_address_from_image(image_path)\n\nif address:\n  print(address)\nelse:\n  print("No GPS data found in image or geocoding failed.")\n'

In [69]:
####### LOAD FLAG ########
def find_image_by_word(word):
  #print(word)
  for filename in os.listdir(flags_path):
    if word.lower() in filename.lower():  # Case-insensitive search
      full_path = os.path.join(flags_folder_path, filename)
      return full_path

In [165]:
from io import StringIO

################# BURNING THE ADDRESS FUNCTION #################
###################################################
#import os
#!pip install Pillow
#from PIL import Image, ImageDraw, ImageFont


# Replace placeholders with your actual paths


def burn_text(folder_path, font_path):


  ########## GET THE IMAGE DATE ###############
  # Get folder name and date
  file_name = os.path.basename(input_image_path)
  #print(folder_name)
  #print(input_image_path)
  '''
  creation_date = datetime.fromtimestamp(os.path.getctime(input_image_path))
  formatted_date = creation_date.strftime("%A, %B %dth, %Y")
  print(formatted_date)
  '''






  #############################################
  ##### CALL GET ADDRESS AND FLAG FUNCTION #######
  ##print((get_address_from_image(input_image_path)))
  if isinstance(get_address_from_image(input_image_path),tuple) :
    formatted_address, Country, Orientation = get_address_from_image(input_image_path)
    #print(formatted_address, Country)
    #print(isinstance(get_address_from_image(input_image_path),tuple))
  else:
    return None

  image = Image.open(os.path.join(folder_path,filename))

  if str(Orientation) == "Horizontal (normal)":
    font_size = 120
    resize_factor = 0.6
    lower_margin_factor = 1
  elif str(Orientation) == "Rotated 180":
    image = image.transpose(Image.ROTATE_180)
    font_size = 120
    resize_factor = 0.6
    lower_margin_factor = 1
  elif str(Orientation) == "Rotated 90 CW":
    image = image.transpose(Image.ROTATE_270)
    font_size =  120
    resize_factor = 0.7
    lower_margin_factor = 1
  elif str(Orientation) == "Rotated 90 UW":
    image = image.transpose(Image.ROTATE_90)
    font_size =  120
    resize_factor = 0.7
    lower_margin_factor = 1
  else :
    Orientation = "Rotated 90 CW"
    font_size =  120
    resize_factor = 0.5
    lower_margin_factor = 1



  #############################################
  ####### CALL find_image_by_word FUNCTION #######

  #### Exception
  if Country == "Türkiye":
    Country = "Turkey"




  full_path= find_image_by_word(Country)
  print(Country)
  png_overlay = Image.open(full_path)
  _, flag_size = png_overlay.size
  #resize_factor = 0.25
  # Resize the PNG
  #png_overlay = png_overlay.resize((int(png_overlay.width * resize_factor), int(png_overlay.height * resize_factor)))






################################
################################
########## ORIGINAL ############
################################

  img_width, img_height = image.size
  # MArgin
  margin = 20 # 5 pixels by default
########## TEXT & MATT############

### DEFINE TEXT ####
   ##### CENTER TEXT ######
    
  #print(formatted_address)  
  if len(str(formatted_address[0])) > len(str(formatted_address[1])):
    size = len(str(formatted_address[0]))
  else:
    size = len(str(formatted_address[1]))

  centered = [line.center(size) for line in formatted_address]
  cent_address = '\n'.join(centered)

    
    
    
  # Calculate text bbox with margin adjustment
  draw = ImageDraw.Draw(image)
  font = ImageFont.truetype(font_path, font_size)  # Adjust font size and path
  text_bbox = draw.textbbox((0, 0), str(cent_address), font=font)
  #print(Orientation)
  #### If text its too long, reduce the text font####
  if str(Orientation) == "Rotated 90 CW" or str(Orientation) == "Rotated 90 UW":
        #print((text_bbox[2]+margin*20+flag_size),(img_height*9/16))
        #print(font_size)
        if (text_bbox[2]+margin*20+flag_size) > (img_height*9/16):
        # Reduce Fint Size until conditions are met
 
          while True: 
              font_size = font_size -1
              font = ImageFont.truetype(font_path, font_size)  # Adjust font size and path
              text_bbox = draw.textbbox((0, 0), str(cent_address), font=font)
              if (text_bbox[2]+margin*20+flag_size) <= (img_height*9/16):
                break
        #print((text_bbox[2]+margin*20+flag_size),(img_height*9/16))
        #print(font_size)
  else:
    if (text_bbox[2]+margin*20) > img_width:
        if (text_bbox[2]+margin*20) > (img_height*9/16):
        # Reduce Fint Size until conditions are met
 
          while True: 
              font_size = font_size -1
              font = ImageFont.truetype(font_path, font_size)  # Adjust font size and path
              text_bbox = draw.textbbox((0, 0), str(cent_address), font=font)
              if (text_bbox[2]+margin*20) <= (img_height*9/16):
                break      
    
        '''
        font_size = int(font_size*(img_width)//(text_bbox[2]+margin*20))
        font = ImageFont.truetype(font_path, font_size)  # Adjust font size and path
        text_bbox = draw.textbbox((0, 0), str(cent_address), font=font)
        '''
    
  # Define Size fo the Flag

  resize_factor =text_bbox[3]/ flag_size
  png_overlay = png_overlay.resize((int(png_overlay.width * resize_factor), int(png_overlay.height * resize_factor)))


  # Calculate position based on bbox with additional margin
  x_pos = img_width //2 - text_bbox[2]//2 +(png_overlay.size[0])//4 ## - margin*5  # right edge + margin
  y_pos = img_height - text_bbox[3] - margin*3  # bottom edge + margin


#### DEFINE MAT #######
# Define box attributes
  box_opacity =230 ### 0 to 255
  box_radius = 35  # Adjust the corner radius
  box_color = (255, 0, 0, box_opacity)  # Red box (customize the color as needed)
  box_width = text_bbox[2]   # Adjust the width
  box_height = text_bbox[3]  # Adjust the height
  box_x = x_pos  # Center the box horizontally
  box_y = y_pos  # Place the box on top (customize the position)



  country_color= {"Argentina": [(45, 154, 227, box_opacity),(255, 255, 255)],
               "Australia": [(16, 106, 224, box_opacity),(255, 255, 255)],
               "Brazil": [(41, 179, 34, box_opacity),(255, 255, 255)],
               "Cambodia": [(232, 82, 242, box_opacity),(255, 255, 255)],
               "Chile": [(219, 15, 124, box_opacity),(255, 255, 255)],
               "Colombia": [(227, 174, 30, box_opacity),(255, 255, 255)],
               "Costa Rica": [(70, 105, 189, box_opacity),(255, 255, 255)],
               "Egypt": [(173, 134, 24, box_opacity),(255, 255, 255)],
               "Hong Kong": [(235, 53, 40, box_opacity),(255, 255, 255)],
               "India": [(201, 196, 26, box_opacity),(255, 255, 255)],
               "Indonesia":[ (81, 191, 31, box_opacity),(255, 255, 255)],
               "Japan": [(242, 53, 31, box_opacity),(255, 255, 255)],
               "Malaysia": [(222, 88, 27, box_opacity),(255, 255, 255)],
               "Morocco": [(250, 27, 65, box_opacity),(255, 255, 255)],
               "New Zealand": [(3, 125, 173, box_opacity),(255, 255, 255)],
               "Peru":[ (207, 112, 116, box_opacity),(255, 255, 255)],
               "Philippines": [(224, 187, 0, box_opacity),(255, 255, 255)],
               "South Africa": [(163, 83, 18, box_opacity),(255, 255, 255)],
               "Singapore": [(186, 22, 52, box_opacity),(255, 255, 255)],
               "Tanzania": [(17, 189, 94, box_opacity),(255, 255, 255)],
               "Turkey": [(224, 45, 224, box_opacity),(255, 255, 255)],
               "United States":[(29, 135, 181, box_opacity),(255, 255, 255)],
               "Vietnam": [(173, 24, 143, box_opacity),(255, 255, 255)],
               "Patagonia": [(240, 240, 242, box_opacity),(255, 255, 255)],
               "*": [(115, 117, 115, box_opacity),(255, 255, 255)],
               "Qatar": [(115, 117, 115, box_opacity),(255, 255, 255)],
               "Thailand": [(115, 117, 115, box_opacity),(255, 255, 255)],
               "Switzerland": [(115, 117, 115, box_opacity),(255, 255, 255)],
               "Fiji": [(115, 117, 115, box_opacity),(255, 255, 255)],
               "Kenya": [(115, 117, 115, box_opacity),(255, 255, 255)],
               "United Arab Emirates": [(115, 117, 115, box_opacity),(255, 255, 255)],
               "Macao": [(115, 117, 115, box_opacity),(255, 255, 255)]
               }


   # Access value by key
  #print(country_color["Turkey"])
   # Access value by key
  #print(country_color["Turkey
  box_color = country_color[Country][0]
  font_color = country_color[Country][1]
  '''
  try:
    box_color = country_color[Country][0]
    font_color = country_color[Country][1]
  finally:
    box_color = (115, 117, 115, box_opacity)
    font_color = (255, 255, 255)
  '''

# Define the rounded rectangle coordinates
  left_top_x = box_x + box_radius
  left_top_y = box_y + box_radius
  right_top_x = box_x + box_width - box_radius
  right_top_y = box_y + box_radius
  right_bottom_x = box_x + box_width - box_radius
  right_bottom_y = box_y + box_height - box_radius
  left_bottom_x = box_x + box_radius
  left_bottom_y = box_y + box_height - box_radius

# Create a transparent overlay image
  overlay = Image.new("RGBA", image.size, (0, 0, 0, 0))  # Transparent black background
  draw = ImageDraw.Draw(overlay)

############ DRAW MAT ##############

# Draw the colored box on the overlay
  draw.rounded_rectangle((box_x - 4*margin, box_y, box_x + box_width + 3*margin, box_y + box_height),radius=box_radius, fill=box_color)
  image.paste(overlay, (0,0), mask=overlay.convert("RGBA").split()[3])



########## DEFINE TEXT AGAIN TO MAKE SURE IT BURNS ###########

  # Calculate text bbox with margin adjustment
  draw = ImageDraw.Draw(image)
  font = ImageFont.truetype(font_path, font_size)  # Adjust font size and path
  text_bbox = draw.textbbox((0, 0), str(cent_address), font=font)

  # Calculate position based on bbox with additional margin
  x_pos = img_width //2 - text_bbox[2]//2 +(png_overlay.size[0])//4 #- margin*5  # right edge + margin
  y_pos = img_height - text_bbox[3] - margin*3    # bottom edge + margin

  #cent_address = "   dfd \n   ddd"
############ DRAW TEXT ##############
  # Draw text with justification and margin
  draw.text((x_pos, y_pos), str(cent_address), font=font, fill=font_color) # White text






############## FLAG ###########


  # Position the overlay image (adjust coordinates as needed)
  #resize_factor =text_bbox[3]/ flag_size
  #png_overlay = png_overlay.resize((int(png_overlay.width * resize_factor), int(png_overlay.height * resize_factor)))

  overlay_x, overlay_y = (x_pos - png_overlay.width -margin//4,  y_pos) ## img_height - png_overlay.width//2) # bottom edge + margin

  # Paste the overlay onto the background with transparency
  image.paste(png_overlay, (overlay_x, overlay_y), mask=png_overlay.convert("RGBA").split()[3])

  # Copy EXIF
  
  exif = image.info['exif']

  output_path = os.path.join(output_path_date, "dated_("+Country +")_"+Country_folder_name+"_"+ filename)  # Prefix output with "dated_"

  if str(Orientation) == "Horizontal (normal)":
    image.save(output_path,'JPEG',quality=95)
  else:
    if str(Orientation) == "Rotated 180":
        image=image.rotate(180, expand=True)
        image.save(output_path,'JPEG',quality=95, exif = exif)
    elif str(Orientation) == "Rotated 90 CW":

        image=image.rotate(90, expand=True)
        image.save(output_path,'JPEG',quality=95, exif = exif)
    elif str(Orientation) == "Rotated 90 UW":

        image=image.rotate(270, expand=True)
        image.save(output_path,'JPEG',quality=95, exif = exif)
    else :

        image=image.rotate(0, expand=True)
        image.save(output_path,'JPEG',quality=95, exif = exif)



  
  #image.save('/content/drive/MyDrive/Image processing/02 USA/destination.jpg', 'JPEG', exif=exif) 
  # Save image
  #image = image.rotate(90)



In [167]:
################################################################################################
################################################################################################
################################################################################################
# Iterate through each JPG file in the directory

import os
from pathlib import Path
from datetime import datetime
Country_folder_name = "Image Selection"
desired_aspect_ratio = "16:9"
matte_color = (255, 245, 238)
shade_opacity = 30
#flags_path = flags_path
#folder_path = date_input_filepath
font_path = "/Users/josemarimon/Desktop/Frame Python/Fonts/Courier_Prime/CourierPrime-BoldItalic.ttf"  # Choose a suitable font
date_input_filepath = Path("/Users/josemarimon/Desktop/Frame Python/Original Pics/"+(Country_folder_name)+"/")
output_path_date = "/Users/josemarimon/Desktop/Frame Python/Edited/"+(Country_folder_name)+"/"
##matt_image_directory = output_path_date
##output_path_matt = "/Users/josemarimon/Desktop/Frame Python/Fotos eileentu/2024-02-18-08h17/Takeout/Google Photos/"+(Country_folder_name)+"/With Matt"
flags_path = "/Users/josemarimon/Desktop/Frame Python/Flags/"
flags_folder_path = os.path.dirname(flags_path)

#################
#################
i=0
#################
#################
file_count = 0
for root, directories, files in os.walk(date_input_filepath):
    for file in files:
        file_count += 1

file_list = []
for file_name1 in os.listdir(date_input_filepath):
    file_list.append(file_name1)
file_list.sort(key=lambda file_name1: file_name1.lower())
##print(file_list)



for filename in file_list:
  i += 1 

  if filename.endswith(".JPG") or filename.endswith(".jpg"):
 
    input_image_path = os.path.join(date_input_filepath, filename)
    print(input_image_path)

    burn_text(date_input_filepath,font_path)
    #input_image_path_matt = os.path.join(output_path_date, "dated_" + filename)
    #output_image_path_matt = os.path.join(output_path_matt, "date_matted_" + filename)
    ##add_mat_with_shade(input_image_path_matt, output_image_path_matt, desired_aspect_ratio, matte_color, shade_opacity)
    print("Mat added to " +filename +" " + str(i) + "/" +str(file_count) +" " + str(round((i)/file_count*100,1))+"%")

    



/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/1025DD61-859D-4BC0-BEE7-FD42966711F5.JPG
Mat added to 1025DD61-859D-4BC0-BEE7-FD42966711F5.JPG 2/463 0.4%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/2717837b-c754-478b-a68a-c6ee079cd6bd.jpg
Mat added to 2717837b-c754-478b-a68a-c6ee079cd6bd.jpg 3/463 0.6%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/2D2CF97F-9DA9-47E9-97F2-D635DD85D6F5.jpg
Mat added to 2D2CF97F-9DA9-47E9-97F2-D635DD85D6F5.jpg 4/463 0.9%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/43fc4234-ae31-4624-9173-b2b172ebdd36.jpg
Mat added to 43fc4234-ae31-4624-9173-b2b172ebdd36.jpg 5/463 1.1%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/50FB4C18-B9B5-464E-AB1E-971169F935D4.jpg
Mat added to 50FB4C18-B9B5-464E-AB1E-971169F935D4.jpg 6/463 1.3%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/70003662182__130DCBE3-E2AB-497D-AED0-8716EFA562.JP

['Buenos Aires, ', "Argentina (Dec 03,'22)"]
['Buenos Aires, ', "Argentina (Dec 03,'22)"]
Argentina
Mat added to IMG_0994.JPG 41/463 8.9%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_0997.JPG
['Buenos Aires, ', "Argentina (Dec 03,'22)"]
['Buenos Aires, ', "Argentina (Dec 03,'22)"]
Argentina
Mat added to IMG_0997.JPG 42/463 9.1%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_1015.JPG
['Buenos Aires, ', "Argentina (Dec 03,'22)"]
['Buenos Aires, ', "Argentina (Dec 03,'22)"]
Argentina
Mat added to IMG_1015.JPG 43/463 9.3%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_1030.JPG
['San Carlos de Bariloche, ', "Argentina (Dec 04,'22)"]
['San Carlos de Bariloche, ', "Argentina (Dec 04,'22)"]
Argentina
Mat added to IMG_1030.JPG 44/463 9.5%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_1046.JPG
['Chuo City, Tokyo, ', "Japan (Apr 08,'23)"]
['Chuo City, Tokyo, ', "Japan (Apr 08,'23)"]
Japa

['Agra, Agra Division', "India (Oct 01,'23)"]
['Agra, Agra Division', "India (Oct 01,'23)"]
India
Mat added to IMG_1653.JPG 78/463 16.8%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_1671.JPG
['Chuo City, Tokyo, ', "Japan (Apr 12,'23)"]
['Chuo City, Tokyo, ', "Japan (Apr 12,'23)"]
Japan
Mat added to IMG_1671.JPG 79/463 17.1%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_1679.JPG
['Agra, Agra Division', "India (Oct 01,'23)"]
['Agra, Agra Division', "India (Oct 01,'23)"]
India
Mat added to IMG_1679.JPG 80/463 17.3%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_1705.JPG
['El Calafate, ', "Argentina (Dec 12,'22)"]
['El Calafate, ', "Argentina (Dec 12,'22)"]
Argentina
Mat added to IMG_1705.JPG 81/463 17.5%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_1719.JPG
['Agra, Agra Division', "India (Oct 01,'23)"]
['Agra, Agra Division', "India (Oct 01,'23)"]
India
Mat added to IMG_1719.JP

Mat added to IMG_2217.JPG 115/463 24.8%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_2231.JPG
['Marrakesh, ', "Morocco (Sep 06,'22)"]
['Marrakesh, ', "Morocco (Sep 06,'22)"]
Morocco
Mat added to IMG_2231.JPG 116/463 25.1%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_2239.JPG
['Pali, Jodhpur Division', "India (Oct 06,'23)"]
['Pali, Jodhpur Division', "India (Oct 06,'23)"]
India
Mat added to IMG_2239.JPG 117/463 25.3%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_2268.JPG
['Marrakesh, ', "Morocco (Sep 06,'22)"]
['Marrakesh, ', "Morocco (Sep 06,'22)"]
Morocco
Mat added to IMG_2268.JPG 118/463 25.5%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_2300.JPG
['Pudahuel, Santiago Province', "Chile (Dec 21,'22)"]
['Pudahuel, Santiago Province', "Chile (Dec 21,'22)"]
Chile
Mat added to IMG_2300.JPG 119/463 25.7%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_

['İstanbul, Beşiktaş', "Türkiye (Sep 12,'22)"]
['İstanbul, Beşiktaş', "Türkiye (Sep 12,'22)"]
Turkey
Mat added to IMG_3055.JPG 153/463 33.0%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_3084.JPG
['Urayasu, Chiba, ', "Japan (Oct 18,'23)"]
['Urayasu, Chiba, ', "Japan (Oct 18,'23)"]
Japan
Mat added to IMG_3084.JPG 154/463 33.3%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_3098.JPG
['Kyoto, Kyoto, ', "Japan (Apr 19,'23)"]
['Kyoto, Kyoto, ', "Japan (Apr 19,'23)"]
Japan
Mat added to IMG_3098.JPG 155/463 33.5%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_3115.JPG
['Christchurch, ', "New Zealand (Jan 08,'23)"]
['Christchurch, ', "New Zealand (Jan 08,'23)"]
New Zealand
Mat added to IMG_3115.JPG 156/463 33.7%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_3123.JPG
['Kyoto, Kyoto, ', "Japan (Apr 19,'23)"]
['Kyoto, Kyoto, ', "Japan (Apr 19,'23)"]
Japan
Mat added to IMG_3123.JPG 157/463

...

['Nowra DC, City of Shoalhaven', "Australia (Feb 04,'23)"]
['Nowra DC, City of Shoalhaven', "Australia (Feb 04,'23)"]
Australia
Mat added to IMG_5660.JPG 309/463 66.7%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_5670.JPG
['Kamishihoro, Kato District', "Japan (May 13,'23)"]
['Kamishihoro, Kato District', "Japan (May 13,'23)"]
Japan
Mat added to IMG_5670.JPG 310/463 67.0%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_5701.JPG
['Obihiro, Hokkaido, ', "Japan (May 14,'23)"]
['Obihiro, Hokkaido, ', "Japan (May 14,'23)"]
Japan
Mat added to IMG_5701.JPG 311/463 67.2%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_5708.JPG
['Nowra DC, City of Shoalhaven', "Australia (Feb 04,'23)"]
['Nowra DC, City of Shoalhaven', "Australia (Feb 04,'23)"]
Australia
Mat added to IMG_5708.JPG 312/463 67.4%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_5744.JPG
['Sapporo, Hokkaido, ', "Japan (May 15,'23

Mat added to IMG_6442.JPG 345/463 74.5%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_6459.JPG
['Hong Kong, ', "Hong Kong (May 27,'23)"]
['Hong Kong, ', "Hong Kong (May 27,'23)"]
Hong Kong
Mat added to IMG_6459.JPG 346/463 74.7%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_6480.JPG
['Coron, Palawan', "Philippines (Feb 17,'23)"]
['Coron, Palawan', "Philippines (Feb 17,'23)"]
Philippines
Mat added to IMG_6480.JPG 347/463 74.9%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_6489.JPG
['Cape Town, ', "South Africa (Oct 11,'22)"]
['Cape Town, ', "South Africa (Oct 11,'22)"]
South Africa
Mat added to IMG_6489.JPG 348/463 75.2%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_6494.JPG
['Macau, ', "Macao (May 28,'23)"]
['Macau, ', "Macao (May 28,'23)"]
Macao
Mat added to IMG_6494.JPG 349/463 75.4%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_6551.JPG
['Coron,

['San Carlos', "Costa Rica (Oct 20,'22)"]
['San Carlos', "Costa Rica (Oct 20,'22)"]
Costa Rica
Mat added to IMG_7248.JPG 381/463 82.3%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_7270.JPG
['Brinchang, Pahang, ', "Malaysia (Mar 01,'23)"]
['Brinchang, Pahang, ', "Malaysia (Mar 01,'23)"]
Malaysia
Mat added to IMG_7270.JPG 382/463 82.5%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_7272.JPG
['San Carlos', "Costa Rica (Oct 20,'22)"]
['San Carlos', "Costa Rica (Oct 20,'22)"]
Costa Rica
Mat added to IMG_7272.JPG 383/463 82.7%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_7288.JPG
['La Fortuna, ', "Costa Rica (Oct 21,'22)"]
['La Fortuna, ', "Costa Rica (Oct 21,'22)"]
Costa Rica
Mat added to IMG_7288.JPG 384/463 82.9%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_7323.JPG
['Bali, ', "Indonesia (Jun 06,'23)"]
['Bali, ', "Indonesia (Jun 06,'23)"]
Indonesia
Mat added to IMG_7323.JPG 38

['Medellín, ', "Colombia (Nov 02,'22)"]
['Medellín, ', "Colombia (Nov 02,'22)"]
Colombia
Mat added to IMG_8491.JPG 418/463 90.3%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_8544.JPG
['Hội An, Hội An', "Vietnam (Mar 16,'23)"]
['Hội An, Hội An', "Vietnam (Mar 16,'23)"]
Vietnam
Mat added to IMG_8544.JPG 419/463 90.5%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_8562.JPG
['Jakarta, ', "Indonesia (Jun 23,'23)"]
['Jakarta, ', "Indonesia (Jun 23,'23)"]
Indonesia
Mat added to IMG_8562.JPG 420/463 90.7%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_8601.JPG
['Guatapé, ', "Colombia (Nov 03,'22)"]
['Guatapé, ', "Colombia (Nov 03,'22)"]
Colombia
Mat added to IMG_8601.JPG 421/463 90.9%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_8605.JPG
['Duy Xuyên District', "Vietnam (Mar 17,'23)"]
['Duy Xuyên District', "Vietnam (Mar 17,'23)"]
Vietnam
Mat added to IMG_8605.JPG 422/463 91.1%
/Users

['Urayasu, Chiba, ', "Japan (Mar 31,'23)"]
['Urayasu, Chiba, ', "Japan (Mar 31,'23)"]
Japan
Mat added to IMG_9737.JPG 456/463 98.5%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_9778.JPG
['Lima, ', "Peru (Nov 14,'22)"]
['Lima, ', "Peru (Nov 14,'22)"]
Peru
Mat added to IMG_9778.JPG 457/463 98.7%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_9809.JPG
['Tama, Tokyo, ', "Japan (Mar 31,'23)"]
['Tama, Tokyo, ', "Japan (Mar 31,'23)"]
Japan
Mat added to IMG_9809.JPG 458/463 98.9%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_9853.JPG
['Tama, Tokyo, ', "Japan (Mar 31,'23)"]
['Tama, Tokyo, ', "Japan (Mar 31,'23)"]
Japan
Mat added to IMG_9853.JPG 459/463 99.1%
/Users/josemarimon/Desktop/Frame Python/Original Pics/Image Selection/IMG_9876.JPG
['Rio de Janeiro, ', "Brazil (Nov 16,'22)"]
['Rio de Janeiro, ', "Brazil (Nov 16,'22)"]
Brazil
Mat added to IMG_9876.JPG 460/463 99.4%
/Users/josemarimon/Desktop/Frame Python/